# Mingrid - Gaming RL Project

### Problem Definition
---
Minigrid is game that that the goal is to navigate a square grid avoid optiscals and reach a exit to the next map. Obstacles is made up of lava and walls. The levels are generated by a random generator in training and the model is tested on a set of pre-made levels.

The main problem question is: Can an RL agent trained on randomly generated MiniGrid environments generalize well enough to solve previously unseen fixed levels?

Rules defined by the game:
-   **Levels**: In testing is 6x6, 7x7, 9x9, 11x11 and 16x16 grids. The training levels are 10x10 grids
-   **Character movement**: Left, Right and Forward. Each action counting as a step (add in that we limited the character to this)
-   **Obstacles**: Walls limit movement and lava ends game with no reward
-   **Field of view**: The grid of view that the charecter sees is a 7x7 grid, where the character is placed in the middle of a edge of the grid. Only seeing forward and to the sides
-   **Goal**: Is to reach green square that will move the charecter to next level

Problems for the model to solve is:
-   How to move the character, learn the inputs it can make
-   Learn how those movement inputs affect the reward
-   Use its field of view to identify the objects in that space
-   Find a strategy move through the level 
-   With the overarching goal to reach the exit as fast as possible

### Limitations
---
The action space is restricted to only movment left, right, forward. What is removed object-interaction complexity (e.g. pickup, toggle), which are not present in the environments. Training is limtied to use 10x10 leval grids that is random genrated. Where the test set is fixed set of 5 levels made from 6x6, 7x7, 9x9, 11x11 and 16x16 grids. 

Reward function and observatin space is predefiende by `MiniGridEnv` and is deafined thouther down here.


### Reward function
---
The reward function is inherited directly from `MiniGridEnv`. This function only rewards the agent for successfuly reaching the goal, the green tile in the grid. 
The terminal reward is time-scaled to the number of steps taken in the level by the agant in each episod. It is defined by:

$$
reward = 1 - \frac{steps\_taken}{max\_steps}
$$

Where `steps_taken` is number of moves in the enviroment made by the agent and `max_step` is the  max limite steps to be made. By this reward defintion the agent gets higher reward for succsefully completing the level as fast as possible, least amount of steps taken witin a episod. When failing to complet the level the reward is zero. The reward function is strictly spares due to no intermediate reawards or peneltys.  

## Agent Structure

---

The agent interacts with the environment through its **observation space**, **action space**, and **reward function**. The reward function is as previously defined. The agent uses **`CnnLstmPolicy`** from Stable Baselines3, which combines a convolutional feature extractor (CNN) with a recurrent memory module (LSTM) and separate policy and value networks. The agent is trained using **Recurrent PPO**, enabling it to make decisions based on both current observations and past experience.

Observations are provided as **image-based tensors** via the `ImgObsWrapper`, with a grid size of 7×7. Each grid cell is encoded as a 3-channel integer vector representing **object type, color, and object state**. This forms a **spatial representation** of the environment, which is processed by the CNN as an image.

Because the environment is **partially observable**, the agent benefits from a recurrent policy that can maintain an internal memory of past observations.

The overall agent architecture consists of:

1. **Observation input:** 7×7×3 image from the MiniGrid environment.
2. **Feature extractor:** Custom CNN (`MinigridFeaturesExtractor`) that converts the image into a 128-dimensional feature vector.
3. **Recurrent layer (LSTM):** Processes sequences of feature vectors over time, allowing the agent to retain temporal information.
4. **Policy network (actor):** Fully connected layers that map the LSTM output to probabilities over three discrete actions (left, right, forward).
5. **Value network (critic):** Fully connected layers that map the LSTM output to a scalar value estimating expected return.
6. **Recurrent PPO algorithm:** Trains the policy and value networks using advantage estimation, clipped policy updates, and sequence-based rollouts.



### Feature Extraction Network (CNN)

The CNN processes image observations and extracts spatial features:

- **Input:** 7×7×3 image tensor  
- **Conv2D layer 1:** 16 filters, kernel size 2×2, activation `ReLU`  
- **Conv2D layer 2:** 32 filters, kernel size 2×2, activation `ReLU`  
- **Conv2D layer 3:** 64 filters, kernel size 2×2, activation `ReLU`  
- **Flatten:** Converts 3D feature maps into a 1D vector  
- **Fully connected layer:** Output dimension = 128, activation `ReLU`  

**Output:** 128-dimensional latent feature vector representing the current observation.



### Recurrent Policy and Value Networks

The PPO agent uses a **recurrent policy architecture**:

- **LSTM layer:** Receives the 128-dimensional CNN features and maintains a hidden state across timesteps, allowing the agent to integrate information over time.
- **Policy network:** Maps the LSTM output to a probability distribution over discrete actions.
- **Value network:** Maps the LSTM output to a scalar value estimating expected return.

The CNN and LSTM layers are shared between the policy and value networks, while the final output layers have separate parameters.



### Training Process

---

Training environments are generated procedurally using a custom `ProceduralLevel` environment with fixed grid size (10×10) and varying obstacle configurations.
The agent is trained using **Proximal Policy Optimization (PPO)** in combination with a CNN-based feature extractor (`MinigridFeaturesExtractor`). Using the CNN–LSTM policy (`CnnLstmPolicy`).

The training process involves the following steps:

### 1. Parallel Environments

- Multiple instances of the environment are created using `EnvOptimiser` and wrapped with `ImgObsWrapper`.  
- Each environment runs independently in parallel, allowing the agent to collect diverse experiences simultaneously.  
- This increases sample efficiency and stabilizes training.

- Training is performed using **multiple environment instances running in parallel** via `SubprocVecEnv`.

- Each environment instance is initialized with a **unique random seed**, ensuring diverse level layouts and state trajectories.

- Parallel execution increases experience diversity, improves sample efficiency, and stabilizes PPO updates.

All environments are wrapped using `ImgObsWrapper`, so the agent consistently receives image-based observations during training.

### 2. Interaction and Data Collection

- At each timestep, the agent receives a 7×7×3 image observation.
- The CNN extracts spatial features, which are passed to the LSTM.
- The LSTM combines current features with its internal memory.
- The **policy network** samples an action based on the LSTM output.
- The environment executes the action and returns the next observation, reward, and termination signal.
- Transitions are collected as **sequences**, preserving temporal order for recurrent learning.

Rollouts are collected across all parallel environments.



### 3. Advantage Estimation and Network Updates

- Recurrent PPO computes **advantages** using value predictions from the critic.
- Policy updates use **clipped objective functions** to limit large policy changes.
- Value updates minimize the error between predicted and observed returns.
- Training is performed on **sequence batches**, ensuring that LSTM hidden states are handled correctly.



### 4. Progress Monitoring and Checkpointing

- A custom callback (`custom_callback`) monitors the **mean episode reward** over a sliding window of the last 100 episodes.
- The model achieving the highest mean reward during training is automatically saved.
- The best-performing model is later used for inference and evaluation.

Overall, the training process allows the agent to learn both **spatial** and **temporal** patterns.

## Inference / Testing
---
Inference and testing are performed in single, non-vectorized environments using pre-defined levels implemented in `MiniGridLevelsEnv`. These levels are specified in a JSON file and represent structured navigation problems of varying grid sizes. These levals have more strutctured problem to solve, rather then being random. This with the intent to be able to show how well the agent have generlized from being trained on random levals. Each episode is run untill it reach maximum number of steps, reaches it goal or steps into lava. 

### Assestment 
This is done by observing, through rendring it's movements. Visual inference is used for qualitative evaluation and debugging, providing insight into agent behavior that is not captured by numerical metrics alone. A successful episode is defined as one in which the agent reaches the goal state before termination. Episodes that end due to time limits or lava contact are counted as failures. The success rate metric does not account for time efficiency, only task completion.

The definition of Succses rate:
$$
\text{Success Rate} = \frac{N_{\text{success}}}{N_{\text{episodes}}}
$$

On top of visual observations this is meant to give degree of answer to how well the agent generalized.